In [2]:
"""
mv_longrun.py -- Long-horizon convergence experiment (defense material)
=======================================================================
Standalone: all the thesis code it needs (data pipeline, calibration,
model, simulator) is included below, copied verbatim from
mv_multiasset_corr.py. Same algorithm, same hyperparameters; the ONLY
change is the number of training iterations N.

Usage:
  python mv_longrun.py --n 200000              # 10x the thesis run
  python mv_longrun.py --n 200000 --resume     # continue after interruption
  python mv_longrun.py --test                  # quick synthetic self-test

The script measures the real speed of YOUR machine on the first 300
iterations and prints the estimated total time. A checkpoint is saved
every --ckpt-every iterations (mv_longrun_ckpt.pkl): the run can be
interrupted at any moment and resumed with --resume.

Outputs:
  mv_longrun_results.npz   phi1 history, benchmarks, metadata
  figure_longrun.pdf/png   convergence of the 5 components + gap vs N
"""

import argparse
import pickle
import time
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm, poisson

# =============================================================================
# THESIS CODE (verbatim from mv_multiasset_corr.py)
# =============================================================================

TICKERS    = ['^GSPC', 'AGG', 'VGK', 'GLD', 'QQQ']
NAMES      = ['S&P 500', 'Bonds', 'MSCI Europe (VGK)', 'Gold', 'NASDAQ-100']
D          = len(TICKERS)   # numero di asset

R_F        = 0.02
DT         = 1 / 12        # passo mensile
T          = 1.0
K          = 12             # mesi per traiettoria
THETA      = 0.1
Z          = 1.4            # target ricchezza
X0         = 1.0

N_TRAIN    = 20000
BATCH      = 32
ALPHA_PSI  = 0.001
ALPHA_OMEGA = 0.05
DECAY      = 0.51
GRAD_CLIP  = 5.0

# Adam per phi
ADAM_LR    = 0.001
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS   = 1e-8

TRAIN_END  = '2019-12-31'
TEST_END   = '2023-12-31'

ETA_TIKHONOV = 0.10   # Tikhonov regularization level (eq. 7.15 della tesi)


def load_multiasset(tickers=TICKERS, train_end=TRAIN_END, test_end=TEST_END):
    """
    Scarica dati mensili per tutti gli asset e restituisce
    matrici di rendimenti log-mensili (train e test).
    """
    import yfinance as yf
    import pandas as pd

    print(f"  Scaricamento dati per {len(tickers)} asset...")
    data = yf.download(tickers, start='2000-01-01', end=test_end,
                      progress=False)
    # Handle both MultiIndex and flat columns
    if isinstance(data.columns, pd.MultiIndex):
        raw = data['Close']
    else:
        raw = data[['Close']].rename(columns={'Close': tickers[0]}) if len(tickers)==1 else data
    raw = raw[list(tickers)]          # FIX: yfinance ordina le colonne alfabeticamente;
                                     # riallineale all'ordine di tickers/NAMES
    # Forward fill brevi gap, poi tronca al primo giorno in cui
    # TUTTI i ticker hanno dati (evita di perdere anni per un ETF)
    raw = raw.ffill().bfill()
    # Trova la data in cui tutti i ticker sono disponibili
    first_valid = raw.apply(lambda col: col.first_valid_index()).max()
    raw = raw.loc[first_valid:]
    print(f"  Dati disponibili dal: {raw.index[0].strftime('%Y-%m-%d')}")

    # Rendimenti log-giornalieri
    log_daily = np.log(raw / raw.shift(1)).dropna()

    # Aggrega in mensili (21 giorni di trading)
    dpm = 21
    n_months = len(log_daily) // dpm
    ret_monthly = np.zeros((n_months, len(tickers)))
    dates_monthly = []
    for i in range(n_months):
        block = log_daily.iloc[i*dpm:(i+1)*dpm]
        ret_monthly[i] = block.sum().values
        dates_monthly.append(pd.Timestamp(block.index[-1]))
    dates_monthly = np.array(dates_monthly)

    # Split train/test
    cutoff = pd.Timestamp(train_end)
    mask_train = np.array([d <= cutoff for d in dates_monthly])
    mask_test  = ~mask_train

    ret_train   = ret_monthly[mask_train]
    ret_test    = ret_monthly[mask_test]
    dates_train = dates_monthly[mask_train]
    dates_test  = dates_monthly[mask_test]

    print(f"  Training: {len(ret_train)} mesi "
          f"({dates_train[0].strftime('%Y-%m')} -> "
          f"{dates_train[-1].strftime('%Y-%m')})")
    print(f"  Test:     {len(ret_test)} mesi "
          f"({dates_test[0].strftime('%Y-%m')} -> "
          f"{dates_test[-1].strftime('%Y-%m')})")
    return ret_train, ret_test, dates_train, dates_test


# ─────────────────────────────────────────────────────────────────────────────
# 3.  CALIBRAZIONE PARAMETRI
# ─────────────────────────────────────────────────────────────────────────────

def calibrate_bs_multi(ret_monthly, r_f=R_F):
    """
    Calibra parametri BS multi-asset dai rendimenti mensili.
    Restituisce mu, Sigma (matrice covarianza diffusiva).
    """
    # Stima annualizzata
    mu    = ret_monthly.mean(axis=0) / DT + 0.5 * (ret_monthly.std(axis=0) / np.sqrt(DT))**2
    Sigma = np.cov(ret_monthly.T) / DT   # matrice covarianza annualizzata
    rho   = mu - r_f                      # vettore excess return
    print(f"  BS mu:  {mu}")
    print(f"  BS rho: {rho}")
    return dict(mu=mu, Sigma=Sigma, rho=rho, SigmaJ=np.zeros((D,D)))


def calibrate_mjd_multi(ret_monthly, r_f=R_F):
    """
    Calibra parametri MJD per ogni asset indipendentemente,
    poi costruisce Sigma dalla componente diffusiva.
    """
    bs = calibrate_bs_multi(ret_monthly, r_f)
    mu_vec    = bs['mu'].copy()
    sigmaJ_sq = np.zeros(D)
    sigma_diff = np.zeros(D)

    for i in range(D):
        ri = ret_monthly[:, i]
        def neg_ll(params):
            mu_i, sig_i, lam_i, m_i, delta_i = params
            if sig_i <= 0 or lam_i <= 0 or delta_i <= 0:
                return 1e10
            kappa = np.exp(m_i + 0.5*delta_i**2) - 1
            n_max = max(10, int(lam_i*DT*5 + 20))
            ns    = np.arange(0, n_max+1)
            pn    = poisson.pmf(ns, lam_i*DT)
            pdf   = np.zeros(len(ri))
            for n, p_n in zip(ns, pn):
                if p_n < 1e-15: continue
                mu_n  = (mu_i - 0.5*sig_i**2 - lam_i*kappa)*DT + n*m_i
                var_n = sig_i**2*DT + n*delta_i**2
                pdf  += p_n * norm.pdf(ri, mu_n, np.sqrt(var_n))
            return -np.sum(np.log(np.maximum(pdf, 1e-300)))

        sig0 = np.std(ri) / np.sqrt(DT)
        x0   = [mu_vec[i], sig0, 5.0, -0.01, 0.03]
        bds  = [(None,None),(1e-4,None),(1e-4,None),(None,None),(1e-4,None)]
        try:
            res = minimize(neg_ll, x0, method='L-BFGS-B', bounds=bds,
                           options=dict(maxiter=300, ftol=1e-8))
            mu_i, sig_i, lam_i, m_i, delta_i = res.x
        except Exception:
            mu_i, sig_i = mu_vec[i], sig0
            lam_i, m_i, delta_i = 5., -0.01, 0.03
        # Fallback: if MJD diffusive vol collapses to < 30% of BS vol,
        # the optimizer moved all variance into jumps — revert to BS vol
        sig_bs_i = np.std(ret_monthly[:, i]) / np.sqrt(DT)
        if sig_i < 0.30 * sig_bs_i:
            sig_i = sig_bs_i
            print(f"    [fallback to BS vol for {NAMES[i]}]")
        sigma_diff[i] = sig_i
        sigmaJ_sq[i]  = lam_i*(np.exp(2*m_i+2*delta_i**2)
                                - 2*np.exp(m_i+0.5*delta_i**2) + 1)
        mu_vec[i] = mu_i
        print(f"  MJD asset {NAMES[i]}: mu={mu_i:.4f}  sigma={sig_i:.4f}"
              f"  lam={lam_i:.2f}  sigmaJ2={sigmaJ_sq[i]:.6f}")

    # Diffusive covariance: use empirical correlations, MJD diffusive vols
    corr  = np.corrcoef(ret_monthly.T)
    Sigma = np.outer(sigma_diff, sigma_diff) * corr
    # SPD projection
    Sigma = 0.5*(Sigma + Sigma.T)
    ev, evec = np.linalg.eigh(Sigma)
    ev = np.maximum(ev, 1e-8)
    Sigma = evec @ np.diag(ev) @ evec.T
    SigmaJ = np.diag(sigmaJ_sq)
    rho   = mu_vec - r_f
    return dict(mu=mu_vec, Sigma=Sigma, rho=rho, SigmaJ=SigmaJ,
                sigma_diff=sigma_diff, sigmaJ_sq=sigmaJ_sq)


def phi1_star_multi(params):
    """
    phi1* = (Sigma + SigmaJ + eps*I)^{-1} rho
    Tikhonov regularization with eta=0.10 (Remark rem:tikhonov, eq. 7.15).
    """
    Sigma_eff = params['Sigma'] + params['SigmaJ']
    eta = ETA_TIKHONOV                              # 0.10
    eps = eta / D * np.trace(Sigma_eff)
    return np.linalg.solve(Sigma_eff + eps*np.eye(D), params['rho'])


# ─────────────────────────────────────────────────────────────────────────────
# 4.  SIMULATORI MULTI-ASSET
# ─────────────────────────────────────────────────────────────────────────────

class RealDataSimulatorMulti:
    """Campiona rendimenti mensili reali con reimmissione."""
    def __init__(self, ret_monthly):
        self.returns = ret_monthly  # shape (n_months, D)

    def simulate_trajectory(self, n=K):
        idx  = np.random.randint(0, len(self.returns), size=n)
        return self.returns[idx]   # shape (n, D)



class MVModelMulti:
    """
    Parametrizzazioni multi-asset (Capitolo 7):
      J^psi(t,x) = (x-omega)^2 * exp(-psi3*(T-t)) + psi2*(t^2-T^2) + psi1*(t-T)
      q^phi(t,x,a) = -exp(-phi_{d+1} - phi_{d+2}*(T-t))/2 * ||a + phi1*(x-omega)||^2
                     - d*theta/2 * [log(2*pi*theta) + phi_{d+1} + phi_{d+2}*(T-t)]
      pi^phi(.|t,x) = N(-phi1*(x-omega), theta*exp(phi_{d+1}+phi_{d+2}*(T-t))*I_d)
    """
    def __init__(self, d=D, psi3_init=0.):
        self.d     = d
        self.psi   = np.array([0., 0., psi3_init])          # (psi1, psi2, psi3)
        self.phi1  = np.zeros(d)                              # vettore actor
        self.phi2  = 0.                                       # log-var intercept
        self.phi3  = 0.                                       # log-var slope
        self.omega = Z

    def _logvar(self, t):
        return np.clip(self.phi2 + self.phi3*(T-t), -20, 20)

    def _c(self, t):
        return np.exp(-self._logvar(t))

    def J(self, t, x):
        p = self.psi; o = self.omega
        e = np.exp(np.clip(-p[2]*(T-t), -30, 30))
        return (x-o)**2*e + p[1]*(t**2-T**2) + p[0]*(t-T) - (o-Z)**2

    def grad_J(self, t, x):
        p = self.psi; o = self.omega
        e = np.exp(np.clip(-p[2]*(T-t), -30, 30))
        return np.array([t-T, t**2-T**2, -(T-t)*(x-o)**2*e])

    def q(self, t, x, a):
        """a: vettore R^d"""
        o   = self.omega
        c   = self._c(t)
        r   = a + self.phi1*(x-o)          # residuo vettoriale
        lv  = self._logvar(t)
        return (-c/2 * np.dot(r, r)
                - self.d*THETA/2*(np.log(2*np.pi*THETA) + lv))

    def grad_q(self, t, x, a):
        """Restituisce (grad_phi1, grad_phi2, grad_phi3)."""
        o   = self.omega
        c   = self._c(t)
        r   = a + self.phi1*(x-o)
        lv  = self._logvar(t)
        rr  = np.dot(r, r)
        g1  = c * r * (x-o)                          # shape (d,)
        g2  = 0.5*c*rr - self.d*THETA/2              # scalar
        g3  = (T-t)*(0.5*c*rr - self.d*THETA/2)     # scalar
        return g1, g2, g3

    def sample_action(self, t, x):
        """Campiona a ~ pi^phi(.|t,x)."""
        mu_a = -self.phi1 * (x - self.omega)
        std_a = np.sqrt(THETA * np.exp(np.clip(self._logvar(t), -20, 20)))
        std_a = max(std_a, 1e-6)
        return np.random.normal(mu_a, std_a)   # shape (d,)

    def mean_action(self, t, x):
        return -self.phi1 * (x - self.omega)   # shape (d,)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  TRAINING OFFLINE MULTI-ASSET

# =============================================================================
# LONG-RUN EXPERIMENT
# =============================================================================

CKPT_FILE = 'mv_longrun_ckpt.pkl'
OUT_NPZ   = 'mv_longrun_results.npz'


# ─────────────────────────────────────────────────────────────────────────────
# Training loop: identical to offline_training_multi in mv_multiasset_corr.py,
# plus: configurable N, timing, periodic checkpoints, resume.
# ─────────────────────────────────────────────────────────────────────────────

def train_long(sim, psi3_init, phi1_star_vec, n_train,
               ckpt_every=5000, resume=False):

    start_j = 0
    if resume and os.path.exists(CKPT_FILE):
        with open(CKPT_FILE, 'rb') as f:
            ck = pickle.load(f)
        if ck['n_train'] != n_train:
            print(f"  [resume] checkpoint was for N={ck['n_train']}, "
                  f"extending/continuing to N={n_train}")
        model = MVModelMulti(d=D, psi3_init=psi3_init)
        model.psi   = ck['psi'];   model.phi1 = ck['phi1']
        model.phi2  = ck['phi2'];  model.phi3 = ck['phi3']
        model.omega = ck['omega']
        m_phi1, v_phi1 = ck['m_phi1'], ck['v_phi1']
        m_phi2, v_phi2 = ck['m_phi2'], ck['v_phi2']
        m_phi3, v_phi3 = ck['m_phi3'], ck['v_phi3']
        adam_step = ck['adam_step']
        start_j   = ck['j']
        np.random.set_state(ck['rng_state'])
        phi1_hist = np.zeros((n_train, D), dtype=np.float32)
        hist_prev = ck['phi1_hist']
        phi1_hist[:min(start_j, n_train)] = hist_prev[:min(start_j, n_train)]
        print(f"  [resume] restarting from iteration {start_j+1}")
    else:
        model = MVModelMulti(d=D, psi3_init=psi3_init)
        phi1_hist = np.zeros((n_train, D), dtype=np.float32)
        m_phi1 = np.zeros(D); v_phi1 = np.zeros(D)
        m_phi2 = 0.;          v_phi2 = 0.
        m_phi3 = 0.;          v_phi3 = 0.
        adam_step = 0

    t0 = time.time()
    t_mark = None

    for j in range(start_j + 1, n_train + 1):
        lr    = j**(-DECAY)
        d_psi = np.zeros(3)
        d_phi1 = np.zeros(D); d_phi2 = 0.; d_phi3 = 0.
        d_omega = 0.

        for _ in range(BATCH):
            rets = sim.simulate_trajectory(K)
            xs   = np.zeros(K+1)
            acts = np.zeros((K, D))
            xs[0] = X0
            for k in range(K):
                acts[k] = model.sample_action(k*DT, xs[k])
                r_excess = np.exp(rets[k]) - 1 - R_F*DT
                xn = xs[k] + np.dot(acts[k], r_excess)
                xs[k+1] = xn if np.isfinite(xn) else xs[k]
            for k in range(K):
                td = (model.J((k+1)*DT, xs[k+1])
                      - model.J(k*DT, xs[k])
                      - model.q(k*DT, xs[k], acts[k])*DT)
                if np.isfinite(td):
                    d_psi  += model.grad_J(k*DT, xs[k]) * td / K
                    g1, g2, g3 = model.grad_q(k*DT, xs[k], acts[k])
                    d_phi1 += g1 * td / K
                    d_phi2 += g2 * td / K
                    d_phi3 += g3 * td / K
            d_omega += xs[K] - Z

        norm_psi = np.linalg.norm(d_psi)
        if norm_psi > GRAD_CLIP * BATCH:
            d_psi *= GRAD_CLIP * BATCH / norm_psi
        norm_phi = np.sqrt(np.dot(d_phi1, d_phi1) + d_phi2**2 + d_phi3**2)
        if norm_phi > GRAD_CLIP * BATCH:
            scale = GRAD_CLIP * BATCH / norm_phi
            d_phi1 *= scale; d_phi2 *= scale; d_phi3 *= scale

        model.psi += lr * ALPHA_PSI * d_psi / BATCH

        adam_step += 1
        g1b = d_phi1 / BATCH; g2b = d_phi2 / BATCH; g3b = d_phi3 / BATCH
        m_phi1 = ADAM_BETA1*m_phi1 + (1-ADAM_BETA1)*g1b
        v_phi1 = ADAM_BETA2*v_phi1 + (1-ADAM_BETA2)*g1b**2
        m_phi2 = ADAM_BETA1*m_phi2 + (1-ADAM_BETA1)*g2b
        v_phi2 = ADAM_BETA2*v_phi2 + (1-ADAM_BETA2)*g2b**2
        m_phi3 = ADAM_BETA1*m_phi3 + (1-ADAM_BETA1)*g3b
        v_phi3 = ADAM_BETA2*v_phi3 + (1-ADAM_BETA2)*g3b**2
        bc1 = 1 - ADAM_BETA1**adam_step
        bc2 = 1 - ADAM_BETA2**adam_step
        model.phi1 += ADAM_LR * (m_phi1/bc1) / (np.sqrt(v_phi1/bc2) + ADAM_EPS)
        model.phi2 += ADAM_LR * (m_phi2/bc1) / (np.sqrt(v_phi2/bc2) + ADAM_EPS)
        model.phi3 += ADAM_LR * (m_phi3/bc1) / (np.sqrt(v_phi3/bc2) + ADAM_EPS)
        model.phi2  = np.clip(model.phi2, -10, 10)
        model.phi3  = np.clip(model.phi3, -10, 10)

        if j % 10 == 0:
            model.omega -= lr * ALPHA_OMEGA * d_omega / (BATCH * 10)

        phi1_hist[j-1] = model.phi1

        # ── timing estimate after 300 fresh iterations ──
        if t_mark is None and j == start_j + 300:
            per_iter = (time.time() - t0) / 300
            eta_h = per_iter * (n_train - j) / 3600
            t_mark = per_iter
            print(f"\n  [timing] {per_iter*1000:.1f} ms/iter on this machine")
            print(f"  [timing] estimated time to N={n_train}: "
                  f"{eta_h:.1f} h ({eta_h*60:.0f} min)\n")

        if j % 2000 == 0:
            gap = np.linalg.norm(phi1_star_vec - model.phi1)
            msg = (f"  Iter {j:7d}/{n_train}  gap={gap:.4f}  "
                   f"phi1={np.round(model.phi1, 3)}")
            if t_mark:
                rem_h = t_mark * (n_train - j) / 3600
                msg += f"  [~{rem_h:.1f} h left]"
            print(msg, flush=True)

        if j % ckpt_every == 0 or j == n_train:
            ck = dict(j=j, n_train=n_train,
                      psi=model.psi, phi1=model.phi1,
                      phi2=model.phi2, phi3=model.phi3, omega=model.omega,
                      m_phi1=m_phi1, v_phi1=v_phi1,
                      m_phi2=m_phi2, v_phi2=v_phi2,
                      m_phi3=m_phi3, v_phi3=v_phi3,
                      adam_step=adam_step,
                      rng_state=np.random.get_state(),
                      phi1_hist=phi1_hist[:j].copy())
            with open(CKPT_FILE, 'wb') as f:
                pickle.dump(ck, f)

    return model, phi1_hist


# ─────────────────────────────────────────────────────────────────────────────
# Analysis and figure
# ─────────────────────────────────────────────────────────────────────────────

def analyze(phi1_hist, phi1_bs, phi1_mjd, n_train):
    w = 300
    sm = np.zeros_like(phi1_hist)
    for i in range(D):
        sm[:, i] = np.convolve(phi1_hist[:, i], np.ones(w)/w, mode='same')

    gaps = np.linalg.norm(phi1_bs[None, :] - phi1_hist, axis=1)

    print("\n" + "="*68)
    print("  GAP ||phi1(N) - phi1*_BS|| AT REFERENCE CHECKPOINTS")
    print("="*68)
    marks = [m for m in [20000, 50000, 100000, 200000, 500000, 1000000]
             if m <= n_train]
    for m in marks:
        print(f"    N = {m:>8d}:  gap = {gaps[m-1]:.4f}   "
              f"phi1 = {np.round(phi1_hist[m-1], 3)}")
    print(f"    (thesis value at N = 20000: 2.20)")
    print(f"    phi1*_BS  = {np.round(phi1_bs, 3)}")
    print(f"    phi1*_MJD = {np.round(phi1_mjd, 3)}")
    print("="*68)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    ax = axes[0]
    colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick', 'purple']
    xs = np.arange(1, n_train+1)
    for i in range(D):
        ax.plot(xs[w:-w], sm[w:-w, i], color=colors[i], lw=1.6, label=NAMES[i])
        ax.axhline(phi1_bs[i], color=colors[i], ls=':', lw=1., alpha=0.7)
    ax.axvline(20000, color='gray', ls='--', lw=1.,
               label='thesis horizon (N = 20000)')
    ax.set_xscale('log')
    ax.set_xlabel('Iteration N (log scale)')
    ax.set_ylabel(r'$\phi_{1,i}$')
    ax.set_title('Components of the learned policy (dotted: BS benchmark)')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.loglog(xs, gaps, color='steelblue', lw=1.4)
    ax.axvline(20000, color='gray', ls='--', lw=1.)
    ax.axhline(2.20, color='gray', ls=':', lw=1.,
               label='thesis gap at N = 20000')
    ax.set_xlabel('Iteration N (log scale)')
    ax.set_ylabel(r'$\|\phi_1(N) - \phi_1^{*,BS}\|$')
    ax.set_title('Euclidean gap to the analytical benchmark')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which='both')

    plt.tight_layout()
    plt.savefig('figure_longrun.pdf', dpi=150, bbox_inches='tight')
    plt.savefig('figure_longrun.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\n  Saved: figure_longrun.pdf/png")


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--n', type=int, default=200000,
                   help='training iterations (thesis: 20000)')
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--ckpt-every', type=int, default=5000)
    p.add_argument('--resume', action='store_true',
                   help='continue from mv_longrun_ckpt.pkl')
    p.add_argument('--test', action='store_true',
                   help='quick self-test on synthetic data (no download)')
    args, _ = p.parse_known_args()

    if args.test:
        print("[TEST MODE] synthetic data, small N")
        rng = np.random.default_rng(0)
        L = np.array([[0.045, 0, 0, 0, 0],
                      [0.002, 0.012, 0, 0, 0],
                      [0.035, 0.002, 0.030, 0, 0],
                      [-0.005, 0.002, 0.005, 0.040, 0],
                      [0.050, 0.001, 0.025, 0.002, 0.030]])
        ret_train = (0.005 + rng.standard_normal((239, 5)) @ L.T)
        ret_test  = (0.005 + rng.standard_normal((48, 5)) @ L.T)
        args.n = min(args.n, 2000)
    else:
        print("[0] Loading data (2000-2023)...")
        ret_train, ret_test, _, _ = load_multiasset()

    print("\n[1] Calibrating benchmarks on the training set...")
    bs_params  = calibrate_bs_multi(ret_train)
    mjd_params = calibrate_mjd_multi(ret_train)
    PHI1_BS  = phi1_star_multi(bs_params)
    PHI1_MJD = phi1_star_multi(mjd_params)
    print(f"  phi1*_BS  = {np.round(PHI1_BS, 3)}")
    print(f"  phi1*_MJD = {np.round(PHI1_MJD, 3)}")

    psi3_init = float(np.mean(np.diag(bs_params['Sigma'])))

    print(f"\n[2] Training EXP2 (diagonal, bootstrap simulator), "
          f"N = {args.n}, seed = {args.seed}")
    if not args.resume:
        np.random.seed(args.seed)
    sim = RealDataSimulatorMulti(ret_train)
    model, phi1_hist = train_long(sim, psi3_init, PHI1_BS, args.n,
                                  ckpt_every=args.ckpt_every,
                                  resume=args.resume)

    print("\n[3] Analysis")
    analyze(phi1_hist, PHI1_BS, PHI1_MJD, args.n)

    np.savez(OUT_NPZ,
             phi1_hist=phi1_hist, phi1_final=model.phi1,
             phi1_bs=PHI1_BS, phi1_mjd=PHI1_MJD,
             n_train=args.n, seed=args.seed)
    print(f"  Saved: {OUT_NPZ}")
    print("\nDone.")

[0] Loading data (2000-2023)...
  Scaricamento dati per 5 asset...
  Dati disponibili dal: 2000-01-03
  Training: 239 mesi (2000-02 -> 2019-12)
  Test:     48 mesi (2020-01 -> 2023-12)

[1] Calibrating benchmarks on the training set...
  BS mu:  [0.05266326 0.03261714 0.04841132 0.07057444 0.08046914]
  BS rho: [0.03266326 0.01261714 0.02841132 0.05057444 0.06046914]
  BS mu:  [0.05266326 0.03261714 0.04841132 0.07057444 0.08046914]
  BS rho: [0.03266326 0.01261714 0.02841132 0.05057444 0.06046914]
  MJD asset S&P 500: mu=0.0518  sigma=0.0668  lam=10.12  sigmaJ2=0.020360
  MJD asset Bonds: mu=0.0326  sigma=0.0208  lam=5.00  sigmaJ2=0.000757


C:\Users\debor\AppData\Local\Temp\ipykernel_5024\929370669.py:161: RuntimeWarning: overflow encountered in exp
  kappa = np.exp(m_i + 0.5*delta_i**2) - 1
c:\Users\debor\anaconda3\Lib\site-packages\scipy\stats\_continuous_distns.py:301: RuntimeWarning: overflow encountered in square
  return np.exp(-x**2/2.0) / _norm_pdf_C


    [fallback to BS vol for MSCI Europe (VGK)]
  MJD asset MSCI Europe (VGK): mu=-0.0051  sigma=0.1682  lam=15.52  sigmaJ2=0.026070
    [fallback to BS vol for Gold]
  MJD asset Gold: mu=0.0836  sigma=0.1627  lam=16.79  sigmaJ2=0.025197
  MJD asset NASDAQ-100: mu=0.0802  sigma=0.1430  lam=3.17  sigmaJ2=0.046011
  phi1*_BS  = [0.292 2.805 0.271 1.845 0.699]
  phi1*_MJD = [ 1.132  2.423 -0.745  1.208  0.873]

[2] Training EXP2 (diagonal, bootstrap simulator), N = 200000, seed = 42

  [timing] 32.6 ms/iter on this machine
  [timing] estimated time to N=200000: 1.8 h (108 min)

  Iter    2000/200000  gap=2.9115  phi1=[0.272 0.14  0.277 0.701 0.448]  [~1.8 h left]
  Iter    4000/200000  gap=2.4488  phi1=[0.254 0.494 0.375 1.044 0.71 ]  [~1.8 h left]
  Iter    6000/200000  gap=2.3158  phi1=[0.477 0.606 0.514 1.211 0.879]  [~1.8 h left]
  Iter    8000/200000  gap=2.3432  phi1=[0.507 0.569 0.511 1.272 0.936]  [~1.7 h left]
  Iter   10000/200000  gap=2.3730  phi1=[0.585 0.54  0.558 1.357 1.002]

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf

TICKERS = ['^GSPC', 'AGG', 'VGK', 'GLD', 'QQQ']

data = yf.download(TICKERS, start='2000-01-01', end='2023-12-31', progress=False)
raw = data['Close'] if isinstance(data.columns, pd.MultiIndex) else data
raw = raw[list(TICKERS)]          # riallinea le colonne (il fix del bug)
raw = raw.ffill().bfill()
first_valid = raw.apply(lambda col: col.first_valid_index()).max()
raw = raw.loc[first_valid:]

log_daily = np.log(raw / raw.shift(1)).dropna()
dpm = 21
n_months = len(log_daily) // dpm
ret_monthly = np.zeros((n_months, 5))
dates_monthly = []
for i in range(n_months):
    block = log_daily.iloc[i*dpm:(i+1)*dpm]
    ret_monthly[i] = block.sum().values
    dates_monthly.append(pd.Timestamp(block.index[-1]))

cutoff = pd.Timestamp('2019-12-31')
mask_train = np.array([d <= cutoff for d in dates_monthly])
ret_train = ret_monthly[mask_train]
ret_test  = ret_monthly[~mask_train]

np.savez('mv_data.npz', ret_train=ret_train, ret_test=ret_test)
print('salvato mv_data.npz  train:', ret_train.shape, ' test:', ret_test.shape)

salvato mv_data.npz  train: (239, 5)  test: (48, 5)
